# 03 - ResNet-18 (from scratch) on PneumoniaMNIST

First of the three architectures being compared (CNN / ViT / hybrid). Implemented from scratch in `src/models.py` (`ResNet18Small`), adapted for small images: the original 7x7 stride-2 stem + max pool is replaced by a single 3x3 stride-1 convolution, since that stem alone would shrink a 28x28 input to 7x7 before any residual block runs. The four residual stages (64/128/256/512 channels) are unchanged from the original ResNet-18 (He et al., 2015).

In [ ]:
import sys
sys.path.append("..")

import torch

from src.data import get_dataloaders
from src.models import build_model
from src.train import train_model, evaluate_on_test
from src.utils import set_seed, get_device

set_seed(42)
device = get_device()
print("Using device:", device)


## 1. Load data

Same subset and normalization strategy as the baseline notebook, so the comparison between models is fair.

In [ ]:
FLAG = "pneumoniamnist"
SIZE = 28
BATCH_SIZE = 128

loaders, spec = get_dataloaders(FLAG, size=SIZE, batch_size=BATCH_SIZE,
                                root="../data", use_train_stats=True)
print(spec)

## 2. Build ResNet-18

In [ ]:
model = build_model("resnet18", n_channels=spec.n_channels, n_classes=spec.n_classes)
n_params = sum(p.numel() for p in model.parameters())
print(f"ResNet-18 (small) has {n_params:,} parameters")

# For reference: the SimpleCNN baseline had ~23.6k parameters.
# ResNet-18 is much larger, so we should expect slower training per epoch
# but likely better capacity to learn complex patterns.

## 3. Train

Same training routine as the baseline (`train_model`), only the model changes.

In [ ]:
history = train_model(
    model, loaders, spec, device,
    epochs=15,
    lr=1e-3,
    out_dir="../results",
    run_name="resnet18_pneumoniamnist",
    monitor="auc",
    early_stopping_patience=5,
)

## 4. Plot training curves

In [ ]:
import matplotlib.pyplot as plt

train_hist, val_hist = history["train"], history["val"]
epochs_ran = [h["epoch"] for h in train_hist]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(epochs_ran, [h["loss"] for h in train_hist], label="train")
axes[0].plot(epochs_ran, [h["loss"] for h in val_hist], label="val")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].legend()
axes[0].set_title("Loss")

axes[1].plot(epochs_ran, [h["auc"] for h in train_hist], label="train")
axes[1].plot(epochs_ran, [h["auc"] for h in val_hist], label="val")
axes[1].axhline(val_hist[0]["majority_baseline_accuracy"], color="gray",
                linestyle="--", label="majority baseline (accuracy)")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("AUC"); axes[1].legend()
axes[1].set_title("AUC")
plt.tight_layout(); plt.show()

## 5. Final evaluation on the test split

In [ ]:
test_result = evaluate_on_test(model, loaders, spec, device,
                               out_dir="../results",
                               run_name="resnet18_pneumoniamnist")
test_result

## 6. Confusion matrix

In [ ]:
cm = test_result.confusion
fig, ax = plt.subplots(figsize=(4, 4))
ax.imshow(cm, cmap="Blues")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha="center", va="center")
labels = [spec.labels[str(c)] for c in range(spec.n_classes)]
ax.set_xticks(range(spec.n_classes)); ax.set_xticklabels(labels)
ax.set_yticks(range(spec.n_classes)); ax.set_yticklabels(labels)
ax.set_xlabel("predicted"); ax.set_ylabel("true")
ax.set_title("Test set confusion matrix")
plt.tight_layout(); plt.show()

## 7. Compare against the SimpleCNN baseline

Loads the baseline's saved test metrics (from notebook 02) for a direct, reproducible comparison instead of copy-pasting numbers by hand.

In [ ]:
import json

with open("../results/simple_cnn_pneumoniamnist_test_metrics.json") as f:
    baseline_metrics = json.load(f)

import pandas as pd
comparison = pd.DataFrame([
    {"model": "SimpleCNN", **baseline_metrics},
    {"model": "ResNet-18", **test_result.as_dict()},
]).set_index("model")
comparison[["accuracy", "balanced_accuracy", "auc", "f1", "sensitivity", "specificity"]]

## 8. Observations

_Write your conclusions here: did ResNet-18 outperform the SimpleCNN baseline? By how much, and was it worth the extra parameters and training time? Did the sensitivity/specificity imbalance improve or stay similar? Any sign of overfitting (large train/val gap)?_